In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

In [ ]:
df=pd.read_csv('adult.csv')

In [ ]:
df.head()

In [ ]:
# Checking the different levels of income
df['income'].value_counts()

In [ ]:
# Converting income to binary class 1 and 0, 1 means the income is greater than 50K, otherwise 0
df.loc[df.income==' <=50K','income']=0
df.loc[df.income==' >50K','income']=1

In [ ]:
# Store variables that needs to convert to dummies, it would be easier if changes are to be made in this process
to_dummy=['workclass','marital_status','occupation','relationship','race','sex','native-country']
newdf=pd.get_dummies(df,columns=to_dummy)

In [ ]:
# Exclude education for now since it's throwing off errors that cannot be coped with at the moment
# Since education-num captures the years of education, we still have information on each individuals' education level
exclude_var=['income','education']
X=newdf.drop(exclude_var,axis=1)
y=newdf['income'].astype('int')

In [ ]:
#Tried this method to deal with earlier errors but did not work
#newdf=pd.to_numeric(newdf.columns[7:], errors='ignore',downcast="integer")

In [ ]:
# Train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=6120)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

#train a very simple model
tree_1 = DecisionTreeClassifier(random_state=0, max_depth=1) #max_depth limits the tree to one split
tree_1.fit(X_train, y_train)



#let's plot it
plot_tree(tree_1, feature_names=X.columns,  
                   class_names=['>50K', '<=50K'],
                      filled=True
                   )

plt.show() #if you remove this it will produce the text behind the tree as well.

In [ ]:
# Use classification report, just a personal preference. It seems everyone is classified as having income less than or equal to 50K
from sklearn.metrics import classification_report,confusion_matrix
print(classification_report(y_train,tree_1.predict(X_train)))
print(classification_report(y_test,tree_1.predict(X_test)))

In [ ]:
# Bagging Model
from sklearn.ensemble import BaggingClassifier

bagging = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100)
bagging.fit(X_train, y_train)

print(classification_report(y_train,bagging.predict(X_train)))
print(classification_report(y_test,bagging.predict(X_test)))

The model definitely seems to be overfit, given the model's perfect performance on train dataset but less desireable performance in the test set, especially a poorer performance in class 1 of target variable (income >50K).

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

boosting = AdaBoostClassifier(DecisionTreeClassifier(), n_estimators=100)
boosting.fit(X_train, y_train)

print(classification_report(y_train,boosting.predict(X_train)))
print(classification_report(y_test,boosting.predict(X_test)))

Same issue spotted in AdaBoost. Let's try to see if we can improve the decision tree with gridsearch.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Start with an initial guess for parameters - with a larger dataset, you'd want many more parameters!
param_grid = {
'max_depth': [5, 10, 15, 20, 25],
'min_samples_split': [10, 20, 30, 40, 50],
'min_impurity_decrease': [0, 0.0005, 0.001, 0.005],
}
gridSearch = GridSearchCV(DecisionTreeClassifier(random_state=6120), param_grid, cv=5,
n_jobs=-1) # n_jobs=-1 will utilize all available CPUs
gridSearch.fit(X_train, y_train)
print('Initial score: ', gridSearch.best_score_)
print('Initial parameters: ', gridSearch.best_params_)

bestRegTree = gridSearch.best_estimator_

In [ ]:
#let's try narrowing in
param_grid = {
'max_depth': [8, 9, 10, 11, 12, 13],
'min_samples_split': [6, 8, 10, 12, 14],
'min_impurity_decrease': [0.0009, 0.00095, 0.001, 0.0015, 0.002],
}

gridSearch = GridSearchCV(DecisionTreeClassifier(random_state=6120), param_grid, cv=5,
n_jobs=-1) # n_jobs=-1 will utilize all available CPUs
gridSearch.fit(X_train, y_train)
print('Improved score: ', gridSearch.best_score_)
print('Improved parameters: ', gridSearch.best_params_)

bestdTree = gridSearch.best_estimator_

In [ ]:
# Best decision tree performance on train and test set
print(classification_report(y_train,bestdTree.predict(X_train)))
print(classification_report(y_test,bestdTree.predict(X_test)))

In [ ]:
#let's plot it, it's very hard to since at greater depth things can get out of hand pretty quickly
#But here's the best combination of fontsize and figsize I found (at least the top depths are readable haha)
plot_tree(bestdTree, feature_names=X.columns,  
                   class_names=['>50K', '<=50K'],
                      filled=True,
          fontsize=6
                   )
plt.figure(figsize=(12,12))  # set plot size (denoted in inches)
plt.show() #if you remove this it will produce the text behind the tree as well.

The decision tree's performance has greatly improved, especially since it is no longer classfying everyone as having income no greater than 50K and the model has ok performance. Still, predicting class 1 seems to be difficult.
Note that the best decision tree has a similar level of performance with bagged model in the test set, but does not suffer from the overfitting issue in train dataset.
Let's try grid search on bagging.

In [ ]:
# candidate parameters for bagging grid search
bag_params = {'n_estimators': [80,100,120],
          'max_samples': [0.5,1.0,1.5],
          'max_features': [0.5,1.0,1.5]}

In [ ]:
# Initialize the gridsearch and fit on training data to see best parameters
bagging_grid = GridSearchCV(BaggingClassifier(DecisionTreeClassifier(),random_state=6120), param_grid =bag_params, cv=3, n_jobs=-1, verbose=1)
bagging_grid.fit(X_train, y_train)
print('Improved score: ', bagging_grid.best_score_)
print('Improved parameters: ', bagging_grid.best_params_)

In [ ]:
#Narrowing down on parameters and one more gridsearch
bag_params = {'n_estimators': [70,80,90],
          'max_samples': [0.75,1.0,1.25],
          'max_features': [0.25,0.5,0.75]}
bagging_grid = GridSearchCV(BaggingClassifier(DecisionTreeClassifier(),random_state=6120), param_grid =bag_params, cv=3, n_jobs=-1, verbose=1)
bagging_grid.fit(X_train, y_train)
print('Improved score: ', bagging_grid.best_score_)
print('Improved parameters: ', bagging_grid.best_params_)

In [ ]:
# The result is the same as the previous run, let's see the performance
best_bagging = bagging_grid.best_estimator_
print(classification_report(y_train,best_bagging.predict(X_train)))
print(classification_report(y_test,best_bagging.predict(X_test)))

The overfitting issue on train dataset is still present, however there is slight improvement for test set performance.